# Avaliação N3 · Modelo Preditivo de Inadimplência

**Análise Preditiva · Engenharia de Software · Católica SC · Prof. Luiz C. Camargo, PhD**

**Alunas:** Maria A. Giuliari e Mariele Vieira · **Apresentação:** 27/11/2026

| Item | O que apresentamos | Pontos | Seção |
|---|---|---|---|
| 1 | Domínio de problema | 1,0 | §1 |
| 2 | Pipeline de dados (Data Lakehouse bronze → prata → ouro) | 2,0 | §2 |
| 3.1 | Três algoritmos + modelo de referência (Dummy) | 2,0 | §3.1 |
| 3.2 | Três métricas e suas características | 2,0 | §3.2 |
| 3.3 | Comparação e escolha justificada | 1,0 | §3.3 |
| 4 | Deploy (pipeline salvo + app Streamlit) | 2,0 | §4 |

### Como executar
1. **Google Colab:** *Arquivo → Fazer upload de notebook* e depois *Ambiente de execução → Executar tudo*.
2. Se o arquivo `credito_sujo.csv` estiver na mesma pasta, ele é usado. Se não estiver, o notebook gera a base com o **mesmo gerador e a mesma semente** da aula (resultado idêntico ao CSV da N2: 3.025 linhas, 16,6% de inadimplentes).
3. Ao final são gerados `modelo_inadimplencia.joblib`, `modelo_meta.json`, `features.py` e `requirements.txt`. Baixe esses arquivos junto com o `app.py` para rodar a interface Streamlit.

## 1 · Domínio de problema

**Contexto:** Uma instituição financeira de Santa Catarina concede empréstimos pessoais. Parte dos clientes atrasa o pagamento por mais de 90 dias e o dinheiro emprestado vira prejuízo. É o mesmo domínio que trabalhamos nas atividades da N2 (pré-processamento e anonimização).

**Problema:** No momento em que o cliente pede o crédito, o analista não sabe se ele vai pagar.

**Objetivo da predição:** Estimar, **no momento da solicitação**, a probabilidade de o cliente se tornar inadimplente, para apoiar a decisão de aprovar, recusar ou mandar para análise manual.

**Variável-alvo:** `inadimplente` (1 = atrasou mais de 90 dias; 0 = pagou em dia).

**Tipo de problema:** **Classificação binária**, com classes **desbalanceadas** (cerca de 17% de inadimplentes).

**Qual erro custa mais caro?**

| Erro | O que acontece | Custo para o banco |
|---|---|---|
| **Falso negativo** (prevê "bom pagador", mas o cliente é inadimplente) | O crédito é aprovado e não é pago | Perde-se o **valor emprestado** (em média ≈ R$ 14 mil por contrato nesta base) |
| **Falso positivo** (prevê "inadimplente", mas o cliente pagaria) | O crédito é recusado | Perde-se apenas a **margem de juros** daquele contrato (e um cliente) |

➡️ **O falso negativo é o erro mais caro.** Por isso as métricas priorizam a capacidade de encontrar os inadimplentes (*recall*) e de ordenar bem o risco, e não a acurácia.

## 2 · Pipeline de dados

O repositório escolhido é um **Data Lakehouse** organizado na **arquitetura medalhão**. Os dados ficam em arquivos **Parquet** (formato colunar e aberto), separados em três camadas:

```
 Sistema de crédito (CSV exportado)
          │  ingestão
          ▼
 🥉 BRONZE  lakehouse/bronze/  → dado bruto, exatamente como chegou (+ metadados de ingestão)
          │  limpeza e tratamento (regras fixas)
          ▼
 🥈 PRATA   lakehouse/prata/   → dado limpo, deduplicado, tipado e padronizado
          │  seleção de atributos válidos + remoção de vazamento
          ▼
 🥇 OURO    lakehouse/ouro/    → tabela pronta para o modelo (só o que existe na hora da decisão)
          │  separação treino/teste
          ▼
 Pipeline scikit-learn: preparar_entrada → imputação → escala/codificação → modelo
```

> Em produção, as mesmas camadas seriam tabelas **Delta Lake** (ou Iceberg) num ambiente como Databricks ou Spark. Aqui reproduzimos a arquitetura localmente com pandas + Parquet, o que é suficiente para o volume da base (3 mil contratos).

In [1]:
import json, platform, warnings
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

SEMENTE = 42          # random_state usado em TODO o notebook
LAKE = Path("lakehouse")
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

scikit-learn 1.9.1 | pandas 3.0.5 | numpy 2.5.3


### 2.0 · Módulo compartilhado `features.py`

As regras de limpeza e a engenharia de atributos ficam num módulo Python separado. O motivo é o **deploy**: o pipeline salvo com joblib guarda apenas uma *referência* às funções. Por isso, o app Streamlit precisa conseguir importar exatamente as mesmas funções usadas no treino.

Isso também resolve o risco que apontamos na reflexão 6.3 da N2: lá, a limpeza ficava **fora** do pipeline. Aqui ela é aplicada **dentro** do pipeline, com `FunctionTransformer`, e um dado novo digitado no app passa pelas mesmas regras do treino.

In [2]:
%%writefile features.py
"""
features.py — regras de limpeza e engenharia de atributos do projeto N3.

Este módulo é importado tanto pelo notebook (treino) quanto pelo app Streamlit (deploy).
Isso é necessário porque o pipeline salvo com joblib guarda apenas uma *referência*
às funções usadas no FunctionTransformer: quem carrega o modelo precisa conseguir
importar `features.preparar_entrada`.

Todas as funções aqui são regras FIXAS, aplicadas linha a linha. Nenhuma delas
aprende estatísticas com os dados (média, mediana, percentil...), portanto
não causam vazamento de dados.
"""
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Colunas usadas pelo modelo (o que existe no momento da concessão do crédito)
# ---------------------------------------------------------------------------
COLUNAS_ENTRADA = [
    "idade", "sexo", "estado_civil", "escolaridade", "renda_mensal", "empregado",
    "tipo_moradia", "mudou_recentemente", "num_veiculos", "dependentes", "cidade",
    "valor_emprestimo", "prazo",
]

NUMERICAS = ["idade", "renda_mensal", "num_veiculos", "dependentes",
             "valor_emprestimo", "prazo", "comprometimento", "log_renda"]
ORDINAIS = ["escolaridade"]
NOMINAIS = ["sexo", "estado_civil", "empregado", "tipo_moradia",
            "mudou_recentemente", "cidade"]
ORDEM_ESCOLARIDADE = [["Fundamental", "Médio", "Superior", "Pós-graduação"]]

# Colunas que NUNCA entram no modelo
#  - id_cliente: identificador, não tem poder preditivo (e é dado pessoal)
#  - data_contrato: usada só para auditoria/validação temporal
#  - qtd_cobrancas: VAZAMENTO DO ALVO (só existe depois do atraso)
COLUNAS_PROIBIDAS = ["id_cliente", "data_contrato", "qtd_cobrancas", "inadimplente"]


def regras_limpeza(d: pd.DataFrame) -> pd.DataFrame:
    """Correções determinísticas (as mesmas da atividade de pré-processamento da N2)."""
    d = d.copy()
    # idade fora do domínio plausível -> ausente (será imputada no pipeline)
    if "idade" in d:
        d["idade"] = pd.to_numeric(d["idade"], errors="coerce")
        d.loc[~d["idade"].between(18, 100), "idade"] = np.nan
    # renda negativa -> ausente
    if "renda_mensal" in d:
        d["renda_mensal"] = pd.to_numeric(d["renda_mensal"], errors="coerce")
        d.loc[d["renda_mensal"] < 0, "renda_mensal"] = np.nan
    # prazo informado em anos -> meses
    if "prazo" in d:
        d["prazo"] = pd.to_numeric(d["prazo"], errors="coerce")
        d["prazo"] = np.where(d["prazo"] < 12, d["prazo"] * 12, d["prazo"])
    # grafias inconsistentes de estado civil
    if "estado_civil" in d:
        d["estado_civil"] = (d["estado_civil"].astype("string").str.strip()
                             .str.capitalize().replace({"Viuvo": "Viúvo"}))
    return d


def criar_atributos(d: pd.DataFrame) -> pd.DataFrame:
    """Engenharia de atributos linha a linha."""
    d = d.copy()
    # fração da renda comprometida com a parcela (sem juros)
    d["comprometimento"] = (d["valor_emprestimo"] / d["prazo"]) / d["renda_mensal"]
    d["comprometimento"] = d["comprometimento"].replace([np.inf, -np.inf], np.nan)
    # log da renda: reduz a assimetria (outliers de 150 e 230 mil)
    d["log_renda"] = np.log1p(d["renda_mensal"])
    return d


def preparar_entrada(d: pd.DataFrame) -> pd.DataFrame:
    """Primeiro passo do pipeline salvo: limpeza + atributos novos.

    Fica DENTRO do pipeline para que um dado novo (ex.: digitado no Streamlit)
    passe exatamente pelas mesmas regras usadas no treino.
    """
    d = regras_limpeza(d)
    d = criar_atributos(d)
    # garante NaN "puro" do numpy nas categóricas (o SimpleImputer não aceita pd.NA)
    for c in NOMINAIS + ORDINAIS:
        if c in d:
            d[c] = d[c].astype(object).where(d[c].notna(), np.nan)
    return d

Overwriting features.py


In [3]:
import importlib, features
importlib.reload(features)
from features import (COLUNAS_ENTRADA, NUMERICAS, ORDINAIS, NOMINAIS, ORDEM_ESCOLARIDADE,
                      COLUNAS_PROIBIDAS, regras_limpeza, preparar_entrada)

### 2.1 · Origem dos dados

A origem é a exportação do sistema de crédito (`credito_sujo.csv`, a mesma base da N2). A função abaixo é o **gerador oficial da aula**, com a mesma semente. Ele só é usado se o CSV não estiver disponível e produz exatamente o mesmo arquivo.

In [4]:
def gerar_credito_sujo(n=3000, seed=7):
    rng = np.random.default_rng(seed)
    cidades = [
        "Florianópolis", "Joinville", "Blumenau", "São José", "Itajaí", "Chapecó",
        "Criciúma", "Jaraguá do Sul", "Palhoça", "Lages", "Balneário Camboriú",
        "Brusque", "Tubarão", "Camboriú", "Navegantes", "São Bento do Sul",
        "Concórdia", "Rio do Sul", "Araranguá", "Gaspar", "Biguaçu", "Indaial",
        "Itapema", "Mafra", "Caçador", "Canoinhas", "Videira", "Içara", "Joaçaba",
        "Xanxerê", "São Miguel do Oeste", "Laguna", "Imbituba", "Tijucas",
        "Guaramirim", "Pomerode", "Timbó", "Porto União", "Curitibanos",
        "Braço do Norte", "Orleans", "Urussanga", "Garopaba", "Bombinhas",
    ]
    peso_cidade = 1 / np.arange(1, len(cidades) + 1) ** 1.1
    peso_cidade /= peso_cidade.sum()

    data = pd.to_datetime("2021-01-01") + pd.to_timedelta(rng.integers(0, 4 * 365, n), unit="D")
    idade = np.clip(rng.normal(40, 12, n), 18, 80).round().astype(int)
    sexo = rng.choice(["F", "M"], n)
    estado_civil = rng.choice(["Casado", "Solteiro", "Divorciado", "Viúvo"], n, p=[0.45, 0.38, 0.12, 0.05])
    escolaridade = rng.choice(["Fundamental", "Médio", "Superior", "Pós-graduação"], n, p=[0.15, 0.42, 0.33, 0.10])
    esc_num = pd.Series(escolaridade).map({"Fundamental": 0, "Médio": 1, "Superior": 2, "Pós-graduação": 3}).to_numpy()
    renda = np.round(np.exp(rng.normal(8.0 + 0.25 * esc_num + 0.006 * (idade - 40), 0.55)), 2)
    empregado = np.where(rng.random(n) < 0.85, "Sim", "Não")
    tipo_moradia = rng.choice(["Própria", "Alugada", "Financiada", "Com os pais"], n, p=[0.38, 0.32, 0.18, 0.12])
    mudou = np.where(rng.random(n) < 0.15, "Sim", "Não")
    num_veiculos = rng.choice([0, 1, 2, 3, 4], n, p=[0.2, 0.45, 0.25, 0.07, 0.03])
    dependentes = rng.choice([0, 1, 2, 3, 4, 5], n, p=[0.35, 0.25, 0.22, 0.1, 0.05, 0.03])
    cidade = rng.choice(cidades, n, p=peso_cidade)
    valor = np.round(np.exp(rng.normal(9.4, 0.6, n)), -2)
    prazo = rng.choice([12, 24, 36, 48, 60], n)

    parcela = valor / prazo
    comprometimento = parcela / renda
    logit = (
        -2.6 + 3.2 * comprometimento - 0.025 * (idade - 40) + 1.1 * (empregado == "Não")
        + 0.18 * dependentes - 0.25 * esc_num + 0.35 * (tipo_moradia == "Alugada")
        + 0.30 * (mudou == "Sim") + 0.25 * (data.year == 2024)
    )
    p = 1 / (1 + np.exp(-logit))
    inadimplente = (rng.random(n) < p).astype(int)
    qtd_cobrancas = np.where(inadimplente == 1, rng.integers(2, 11, n), rng.choice([0, 0, 0, 0, 1], n))

    df = pd.DataFrame({
        "id_cliente": np.arange(100001, 100001 + n), "data_contrato": data.strftime("%Y-%m-%d"),
        "idade": idade.astype(float), "sexo": sexo, "estado_civil": estado_civil,
        "escolaridade": escolaridade, "renda_mensal": renda, "empregado": empregado,
        "tipo_moradia": tipo_moradia, "mudou_recentemente": mudou,
        "num_veiculos": num_veiculos.astype(float), "dependentes": dependentes, "cidade": cidade,
        "valor_emprestimo": valor, "prazo": prazo, "qtd_cobrancas": qtd_cobrancas,
        "inadimplente": inadimplente,
    })

    p_falta_renda = np.where(df.renda_mensal > 9000, 0.40, 0.05)
    df.loc[rng.random(n) < p_falta_renda, "renda_mensal"] = np.nan
    neg = rng.choice(n, 6, replace=False)
    df.loc[neg, "renda_mensal"] = -df.loc[neg, "renda_mensal"].fillna(8700)
    df.loc[rng.choice(n, 3, replace=False), "renda_mensal"] = [95000.0, 150000.0, 230000.0]
    p_falta_emp = np.clip(0.05 + 0.012 * (df.idade - 18), 0, 0.8)
    df.loc[rng.random(n) < p_falta_emp, "empregado"] = np.nan
    bloco = rng.random(n) < 0.035
    df.loc[bloco, ["tipo_moradia", "mudou_recentemente", "num_veiculos"]] = np.nan
    df.loc[rng.random(n) < 0.04, "escolaridade"] = np.nan
    df.loc[rng.choice(n, 8, replace=False), "idade"] = rng.choice([0, 146, -1, 999], 8)
    variantes = {"Casado": ["casado", "CASADO", "Casado "], "Solteiro": ["solteiro", " Solteiro", "SOLTEIRO"],
                 "Divorciado": ["divorciado"], "Viúvo": ["Viuvo", "viúvo"]}
    for original, vs in variantes.items():
        idx = df.index[(df.estado_civil == original) & (rng.random(n) < 0.08)]
        df.loc[idx, "estado_civil"] = rng.choice(vs, len(idx))
    anos = rng.random(n) < 0.03
    df.loc[anos, "prazo"] = df.loc[anos, "prazo"] // 12
    dups = df.sample(25, random_state=7)
    return pd.concat([df, dups], ignore_index=True).sample(frac=1, random_state=7).reset_index(drop=True)

In [5]:
CAMINHO_CSV = Path("credito_sujo.csv")   # no Colab com Drive: Path("/content/drive/MyDrive/.../credito_sujo.csv")

if CAMINHO_CSV.exists():
    origem = pd.read_csv(CAMINHO_CSV)
    fonte = str(CAMINHO_CSV)
else:
    origem = gerar_credito_sujo()        # mesmo conteúdo do credito_sujo.csv
    origem.to_csv(CAMINHO_CSV, index=False)
    fonte = "gerar_credito_sujo(n=3000, seed=7)"

print(fonte, origem.shape)
origem.head()

gerar_credito_sujo(n=3000, seed=7) (3025, 17)


,id_cliente,data_contrato,idade,sexo,estado_civil,escolaridade,renda_mensal,empregado,tipo_moradia,mudou_recentemente,num_veiculos,dependentes,cidade,valor_emprestimo,prazo,qtd_cobrancas,inadimplente
0,102636,2021-11-25,49.0,M,Casado,Superior,2802.74,NaN,Própria,Não,0.0,2,Joinville,21600.0,24,1,0
1,102912,2022-08-18,34.0,M,Solteiro,Médio,724.72,Sim,Com os pais,Não,2.0,3,Florianópolis,9700.0,24,1,0
2,100581,2023-03-27,50.0,F,Solteiro,Médio,15191.51,Sim,Financiada,Não,0.0,0,Videira,17800.0,12,0,0
3,102330,2021-07-05,60.0,M,Divorciado,Superior,5949.22,NaN,Alugada,Não,2.0,1,Palhoça,9900.0,24,0,0
4,102375,2023-07-29,33.0,M,Solteiro,Superior,2914.23,Sim,Alugada,Não,0.0,1,Tubarão,23500.0,24,0,0


### 2.2 · Ingestão → camada **Bronze**

A camada bronze guarda o dado **bruto, sem nenhuma alteração**, com metadados de linhagem (quando e de onde veio). Se uma regra de limpeza estiver errada, dá para reprocessar tudo a partir daqui.

In [6]:
for camada in ["bronze", "prata", "ouro"]:
    (LAKE / camada).mkdir(parents=True, exist_ok=True)

bronze = origem.copy()
bronze["_ingestao_ts"] = pd.Timestamp.now().floor("s")
bronze["_fonte"] = fonte
bronze.to_parquet(LAKE / "bronze" / "contratos_credito.parquet", index=False)

print("Bronze:", bronze.shape)
print("Faltantes (%):", (bronze.isna().mean() * 100).round(1)[lambda s: s > 0].to_dict())

Bronze: (3025, 19)
Faltantes (%): {'escolaridade': 4.1, 'renda_mensal': 8.5, 'empregado': 31.4, 'tipo_moradia': 3.0, 'mudou_recentemente': 3.0, 'num_veiculos': 3.0}


### 2.3 · Limpeza e tratamento → camada **Prata**

Aqui entram apenas correções **determinísticas**, que não aprendem nada com os dados (as mesmas da N2):

| Problema encontrado na AED (N2) | Tratamento |
|---|---|
| 25 linhas duplicadas | `drop_duplicates()` **antes** da separação (se não, a mesma linha poderia cair no treino e no teste, o que é vazamento) |
| `estado_civil` com 13 grafias para 4 categorias | `strip` + `capitalize` + `Viuvo → Viúvo` |
| `idade` = −1, 0, 146, 999 | vira `NaN` (é erro de digitação, não informação; será imputada no pipeline) |
| `renda_mensal` negativa | vira `NaN` |
| `prazo` informado em anos (1 a 5) | converte para meses (× 12) |
| `data_contrato` como texto | converte para `datetime` |

Valores **ausentes não são preenchidos aqui**. A imputação usa estatísticas (mediana, moda), então precisa ser feita **depois** da separação treino/teste, dentro do pipeline.

In [7]:
prata = (bronze.drop(columns=["_ingestao_ts", "_fonte"])
               .drop_duplicates()
               .pipe(regras_limpeza)
               .assign(data_contrato=lambda d: pd.to_datetime(d["data_contrato"]))
               .reset_index(drop=True))
prata.to_parquet(LAKE / "prata" / "contratos_credito.parquet", index=False)

assert prata.duplicated().sum() == 0 and len(prata) == 3000
assert set(prata["estado_civil"].unique()) == {"Casado", "Solteiro", "Divorciado", "Viúvo"}
assert set(prata["prazo"].unique()) <= {12, 24, 36, 48, 60}
print("Prata:", prata.shape, "| taxa de inadimplência:", round(prata["inadimplente"].mean(), 3))

Prata: (3000, 17) | taxa de inadimplência: 0.166


### 2.4 · Transformação → camada **Ouro** (tabela de atributos)

A camada ouro contém **só as colunas que existem no momento da decisão de crédito**, mais o alvo:

- ❌ `qtd_cobrancas` é removida: **vazamento do alvo**. As cobranças só são enviadas *depois* do atraso. Na N2 ela levou a AUC artificialmente para 1,0.
- ❌ `id_cliente` é removido: é identificador, não tem poder preditivo e é dado pessoal (LGPD).
- `data_contrato` fica apenas para auditoria e para a checagem temporal; **não** entra como atributo.

**Engenharia de atributos** (definida em `features.criar_atributos`, aplicada dentro do pipeline):

- `comprometimento = (valor_emprestimo / prazo) / renda_mensal`: fração da renda que vai para a parcela. É o principal sinal de risco em crédito.
- `log_renda = log(1 + renda_mensal)`: reduz a assimetria causada pelos outliers de 150 e 230 mil.

Na N2 esses dois atributos subiram a AUC de 0,708 para 0,722 e a *average precision* de 0,388 para 0,443.

In [8]:
ouro = prata[["data_contrato"] + COLUNAS_ENTRADA + ["inadimplente"]].copy()
ouro.to_parquet(LAKE / "ouro" / "features_inadimplencia.parquet", index=False)

assert not set(["qtd_cobrancas", "id_cliente"]) & set(ouro.columns)
print("Ouro:", ouro.shape)
for camada in ["bronze", "prata", "ouro"]:
    for arq in (LAKE / camada).glob("*.parquet"):
        print(f"  {arq}  ({arq.stat().st_size/1024:.0f} KB)")

Ouro: (3000, 15)
  lakehouse/bronze/contratos_credito.parquet  (78 KB)
  lakehouse/prata/contratos_credito.parquet  (78 KB)
  lakehouse/ouro/features_inadimplencia.parquet  (60 KB)


### 2.5 · Separação dos dados

- **Holdout estratificado**: 75% treino / 25% teste, com `stratify=y` (mantém os ~17% de inadimplentes nas duas partes) e `random_state=42`.
- O conjunto de **teste fica guardado** e só é usado uma vez, na avaliação final.
- Para comparar modelos *dentro* do treino usamos **validação cruzada estratificada (5 folds)**.
- Como checagem extra de robustez (coerente com a N2), também avaliamos uma **divisão temporal**: treino 2021–2023 e teste 2024.

In [9]:
ouro = pd.read_parquet(LAKE / "ouro" / "features_inadimplencia.parquet")   # o modelo lê da camada ouro
X = ouro[COLUNAS_ENTRADA]
y = ouro["inadimplente"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEMENTE)

print(f"Treino: {len(X_treino)} linhas | inadimplência {y_treino.mean():.3f}")
print(f"Teste : {len(X_teste)} linhas | inadimplência {y_teste.mean():.3f}")

Treino: 2250 linhas | inadimplência 0.166
Teste : 750 linhas | inadimplência 0.165


### 2.6 · Pipeline de pré-processamento (sem vazamento)

Tudo que **aprende com os dados** está **dentro** do `Pipeline`. Assim, mediana, média, desvio e categorias são calculados **só com o treino** (e só com o treino de cada fold na validação cruzada) e apenas aplicados no teste.

| Etapa | Colunas | Transformação |
|---|---|---|
| `prepara` | todas | `FunctionTransformer(preparar_entrada)`: regras de limpeza + `comprometimento` e `log_renda` (linha a linha, não aprende nada) |
| numérico | idade, renda, veículos, dependentes, valor, prazo, comprometimento, log_renda | `SimpleImputer(median, add_indicator=True)` → `StandardScaler` |
| ordinal | escolaridade | `SimpleImputer(most_frequent)` → `OrdinalEncoder` (Fundamental < Médio < Superior < Pós) |
| nominal | sexo, estado civil, empregado, moradia, mudou, cidade | `SimpleImputer(constant="desconhecido")` → `OneHotEncoder(handle_unknown="ignore", min_frequency=20)` |

`empregado` tem 31% de faltantes com mecanismo **MAR** (cresce com a idade, como vimos na N2). Por isso o faltante vira a categoria `"desconhecido"`, em vez de chutarmos Sim ou Não.

In [10]:
def criar_preprocessador():
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                          ("esc", StandardScaler())]), NUMERICAS),
        ("ord", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("enc", OrdinalEncoder(categories=ORDEM_ESCOLARIDADE))]), ORDINAIS),
        ("nom", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="desconhecido")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", min_frequency=20,
                                               sparse_output=False))]), NOMINAIS),
    ])

def criar_pipeline(estimador):
    # clone(): cada pipeline recebe uma cópia NOVA do estimador (evita reaproveitar um modelo já treinado)
    return Pipeline([
        ("prepara", FunctionTransformer(preparar_entrada)),
        ("prep", criar_preprocessador()),
        ("clf", clone(estimador)),
    ])

criar_pipeline(LogisticRegression())

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prepara', ...), ('prep', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function pre...x7ffb663f8400>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature_names_out` is not None.See ``get_feature_names_out`` for more details... versionadded:: 1.1",None
,"kw_args kw_args: dict, default=NoneDictionary o

**Resumo dos cuidados contra vazamento de dados**

1. `qtd_cobrancas` (atributo que só existe **depois** do resultado) foi removida já na camada ouro.
2. Duplicatas foram removidas **antes** da separação, para a mesma linha não aparecer no treino e no teste.
3. Imputação, padronização e codificação são ajustadas **depois** da separação, só no treino, dentro do pipeline.
4. A validação cruzada recebe o pipeline **inteiro**, então cada fold reajusta o pré-processamento só com seus dados de treino.
5. Os atributos criados são fórmulas linha a linha (não usam estatística do conjunto), então não vazam informação.
6. O conjunto de teste é usado **uma única vez**, na avaliação final.

## 3 · Modelagem e avaliação

### 3.1 · Treinamento e teste

| Modelo | Por que testar |
|---|---|
| **DummyClassifier** (`most_frequent`) | Referência: sempre prevê "adimplente". Mostra se os modelos aprenderam algo além do óbvio |
| **Regressão Logística** (`class_weight="balanced"`) | Modelo linear, interpretável (coeficientes → *odds ratio*) e padrão de mercado em *credit scoring* |
| **Random Forest** (300 árvores, `min_samples_leaf=10`, `balanced`) | Conjunto de árvores (*bagging*): captura não linearidades e interações |
| **HistGradientBoosting** (`learning_rate=0.05`, `max_depth=3`, `balanced`) | *Boosting* de árvores, geralmente forte em dados tabulares |

`class_weight="balanced"` dá mais peso aos inadimplentes no treino. Sem isso, com 17% de positivos, o modelo tenderia a prever quase sempre "adimplente", justamente o erro mais caro.

**Protocolo de avaliação:** holdout estratificado 75/25 com `random_state=42`. Antes, fazemos uma validação cruzada estratificada de 5 folds **só no treino**. As métricas de treino aparecem apenas para discutir sobreajuste.

In [11]:
modelos = {
    "Dummy (referência)": DummyClassifier(strategy="most_frequent"),
    "Regressão Logística": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=10,
                                            class_weight="balanced", random_state=SEMENTE, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingClassifier(learning_rate=0.05, max_iter=200, max_depth=3,
                                                           class_weight="balanced", random_state=SEMENTE),
}

# Validação cruzada estratificada (5 folds) SOMENTE no treino
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMENTE)
metricas_cv = {"ROC-AUC": "roc_auc", "PR-AUC": "average_precision", "Recall": "recall"}

linhas_cv = {}
for nome, est in modelos.items():
    r = cross_validate(criar_pipeline(est), X_treino, y_treino, cv=cv, scoring=metricas_cv)
    linhas_cv[nome] = {m: f"{r['test_' + m].mean():.3f} ± {r['test_' + m].std():.3f}" for m in metricas_cv}

pd.DataFrame(linhas_cv).T.rename_axis("Validação cruzada (treino, 5 folds)")

,ROC-AUC,PR-AUC,Recall
"Validação cruzada (treino, 5 folds)",,,
Dummy (referência),0.500 ± 0.000,0.166 ± 0.001,0.000 ± 0.000
Regressão Logística,0.718 ± 0.038,0.437 ± 0.036,0.591 ± 0.068
Random Forest,0.716 ± 0.037,0.395 ± 0.030,0.524 ± 0.048
HistGradientBoosting,0.695 ± 0.039,0.397 ± 0.019,0.527 ± 0.050


In [12]:
# Treino no conjunto de treino completo e avaliação ÚNICA no conjunto de teste
treinados, resultados = {}, []
for nome, est in modelos.items():
    pipe = criar_pipeline(est).fit(X_treino, y_treino)
    treinados[nome] = pipe
    proba_te = pipe.predict_proba(X_teste)[:, 1]
    pred_te = pipe.predict(X_teste)
    proba_tr = pipe.predict_proba(X_treino)[:, 1]
    resultados.append({
        "Modelo": nome,
        "ROC-AUC": roc_auc_score(y_teste, proba_te),
        "PR-AUC": average_precision_score(y_teste, proba_te),
        "Recall": recall_score(y_teste, pred_te),
        "Precisão (apoio)": precision_score(y_teste, pred_te, zero_division=0),
        "ROC-AUC treino": roc_auc_score(y_treino, proba_tr),
    })

tabela = pd.DataFrame(resultados).set_index("Modelo")
tabela["Gap AUC (treino−teste)"] = tabela["ROC-AUC treino"] - tabela["ROC-AUC"]
tabela.round(3)

,ROC-AUC,PR-AUC,Recall,Precisão (apoio),ROC-AUC treino,Gap AUC (treino−teste)
Modelo,,,,,,
Dummy (referência),0.500,0.165,0.000,0.000,0.500,0.000
Regressão Logística,0.697,0.416,0.524,0.267,0.761,0.064
Random Forest,0.681,0.384,0.516,0.317,0.919,0.238
HistGradientBoosting,0.657,0.360,0.492,0.288,0.878,0.221


**Leitura sobre sobreajuste (*overfitting*):** a Random Forest e o HistGradientBoosting têm AUC de treino muito maior que a de teste (gap ≈ 0,20 a 0,24). Eles **decoraram** padrões do treino que não se repetem em dados novos. A Regressão Logística tem o menor gap (≈ 0,06), ou seja, generaliza melhor. Com apenas 3 mil linhas e uma relação de risco que é aproximadamente linear no *logit* (comprometimento de renda, emprego, dependentes...), um modelo mais simples é suficiente.

O **Dummy** tem ROC-AUC = 0,5 (chute), PR-AUC ≈ 0,165 (igual à proporção de inadimplentes) e Recall = 0: nunca encontra um inadimplente. Todos os modelos superam essa referência, então **aprenderam algo além do óbvio**. Lembrando a N2: a *acurácia* do Dummy seria ≈ 83%, o que mostra por que acurácia não serve neste problema.

### 3.2 · Métricas de desempenho

As três métricas foram escolhidas pelo **custo do erro**: o falso negativo (aprovar quem não paga) é o mais caro, e as classes são desbalanceadas.

#### 📈 Métrica 1: ROC-AUC (área sob a curva ROC)
- **O que mede:** a capacidade de **ordenar** os clientes por risco. É a probabilidade de o modelo dar um score maior a um inadimplente sorteado do que a um adimplente sorteado.
- **Como é calculada:** para todos os limiares de decisão, calcula-se TPR = VP/(VP+FN) e FPR = FP/(FP+VN). A curva TPR × FPR é desenhada e mede-se a área sob ela.
- **Faixa:** 0 a 1. **0,5 = chute** (Dummy). Em *credit scoring*, acima de 0,70 já é útil e acima de 0,80 é muito bom.
- **Por que serve aqui:** o banco usa o modelo como **score de risco** para montar faixas (aprovar / analisar / recusar). A ROC-AUC avalia o score independentemente do limiar escolhido.
- **Limitação:** com classes desbalanceadas pode parecer otimista, porque os muitos verdadeiros negativos deixam a FPR baixa. Por isso ela vem acompanhada da PR-AUC.

#### 🎯 Métrica 2: PR-AUC / *Average Precision* (robusta ao desbalanceamento)
- **O que mede:** a qualidade do modelo **olhando só para a classe positiva** (inadimplentes): quando ele aponta alguém como arriscado, quantas vezes acerta, ao longo de todos os níveis de *recall*.
- **Como é calculada:** AP = Σₙ (Rₙ − Rₙ₋₁) · Pₙ, ou seja, a média das precisões em cada limiar, ponderada pelo ganho de *recall*.
- **Faixa:** 0 a 1. A **linha de base é a prevalência** (≈ 0,165 aqui, o valor do Dummy). Um valor bom é **bem acima** dela; 0,40 já é cerca de 2,5× a linha de base.
- **Por que serve aqui:** não depende dos verdadeiros negativos, então não é "inflada" pela maioria de bons pagadores. É a métrica que o enunciado pede como robusta a desbalanceamento.

#### 🔍 Métrica 3: Recall (sensibilidade)
- **O que mede:** a fração dos inadimplentes reais que o modelo **consegue identificar**.
- **Como é calculada:** Recall = VP / (VP + FN), no limiar de decisão 0,5.
- **Faixa:** 0 a 1. O Dummy tem 0. Quanto maior, menos calotes passam despercebidos.
- **Por que serve aqui:** mede **diretamente o erro mais caro**. Cada falso negativo é um empréstimo aprovado que vira prejuízo.
- **Cuidado:** prever "inadimplente" para todo mundo daria Recall = 1 e recusaria todos os clientes. Por isso mostramos a **precisão** como coluna de apoio e usamos as duas métricas anteriores, que não dependem de limiar.

### 3.3 · Comparação dos modelos e escolha

In [13]:
comparacao = tabela[["ROC-AUC", "PR-AUC", "Recall"]].round(3)
comparacao.style.highlight_max(axis=0, subset=pd.IndexSlice[comparacao.index[1:], :], color="#c6efce") \
               .format("{:.3f}").set_caption("Conjunto de TESTE (25%, estratificado, random_state=42)")

,ROC-AUC,PR-AUC,Recall
Modelo,,,
Dummy (referência),0.500,0.165,0.000
Regressão Logística,0.697,0.416,0.524
Random Forest,0.681,0.384,0.516
HistGradientBoosting,0.657,0.360,0.492


In [14]:
# Checagem de robustez: divisão TEMPORAL (treina 2021-2023, testa 2024), como na N2
ano = ouro["data_contrato"].dt.year
Xa, ya, Xb, yb = X[ano <= 2023], y[ano <= 2023], X[ano == 2024], y[ano == 2024]
temporal = {}
for nome, est in modelos.items():
    p = criar_pipeline(est).fit(Xa, ya)
    pr = p.predict_proba(Xb)[:, 1]
    temporal[nome] = {"ROC-AUC": roc_auc_score(yb, pr), "PR-AUC": average_precision_score(yb, pr),
                      "Recall": recall_score(yb, p.predict(Xb))}
pd.DataFrame(temporal).T.round(3).rename_axis("Teste temporal (2024)")

,ROC-AUC,PR-AUC,Recall
Teste temporal (2024),,,
Dummy (referência),0.500,0.179,0.000
Regressão Logística,0.722,0.443,0.589
Random Forest,0.713,0.425,0.489
HistGradientBoosting,0.698,0.399,0.489


In [15]:
melhor = "Regressão Logística"
modelo_final = treinados[melhor]

pred = modelo_final.predict(X_teste)
vn, fp, fn, vp = confusion_matrix(y_teste, pred).ravel()
print(f"Matriz de confusão ({melhor}, teste, limiar 0,5)")
print(pd.DataFrame([[vn, fp], [fn, vp]],
                   index=["Real: adimplente", "Real: inadimplente"],
                   columns=["Previsto: adimplente", "Previsto: inadimplente"]))
print(f"\nInadimplentes identificados: {vp} de {vp + fn} (recall = {vp/(vp+fn):.1%})")
print(f"Falsos negativos (o erro caro): {fn}")

Matriz de confusão (Regressão Logística, teste, limiar 0,5)
                    Previsto: adimplente  Previsto: inadimplente
Real: adimplente                     448                     178
Real: inadimplente                    59                      65

Inadimplentes identificados: 65 de 124 (recall = 52.4%)
Falsos negativos (o erro caro): 59


**Escolha: Regressão Logística** ✅

1. **Melhor nas três métricas no teste** (ROC-AUC, PR-AUC e Recall). A validação cruzada no treino aponta o mesmo resultado, então não foi sorte da divisão.
2. **Resultado se mantém na divisão temporal** (2024). O modelo deve funcionar para contratos futuros, que é o uso real.
3. **Menor sobreajuste**: o gap entre treino e teste é o menor dos três algoritmos. Random Forest e Gradient Boosting decoraram o treino.
4. **Domínio de crédito**: é interpretável. Cada coeficiente vira um *odds ratio* que o analista entende, e isso permite **explicar a recusa ao cliente**. A LGPD (art. 20) garante ao titular o direito de pedir revisão e explicação de decisões automatizadas. Um modelo caixa-preta dificultaria isso.

**Limitação honesta:** um ROC-AUC de ~0,70 é um desempenho **moderado**. O modelo serve como **apoio** à decisão (priorizar análise manual dos casos de maior risco), não como decisão automática definitiva.

In [16]:
# Interpretação: os atributos que mais aumentam / reduzem o risco (odds ratio)
nomes = modelo_final.named_steps["prep"].get_feature_names_out()
coef = modelo_final.named_steps["clf"].coef_[0]
interp = (pd.DataFrame({"atributo": nomes, "coeficiente": coef, "odds_ratio": np.exp(coef)})
            .query("~atributo.str.contains('cidade_')")      # cidades: poucos contratos cada, coeficientes ruidosos
            .sort_values("coeficiente", ascending=False))
pd.concat([interp.head(6), interp.tail(5)]).round(3).reset_index(drop=True)

,atributo,coeficiente,odds_ratio
0,nom__empregado_Não,0.924,2.521
1,num__comprometimento,0.600,1.822
2,nom__tipo_moradia_Alugada,0.250,1.285
3,num__dependentes,0.206,1.228
4,nom__estado_civil_Solteiro,0.179,1.196
5,nom__mudou_recentemente_Sim,0.118,1.125
6,num__missingindicator_idade,-0.277,0.758
7,ord__escolaridade,-0.279,0.757
8,nom__empregado_desconhecido,-0.357,0.700
9,num__idade,-0.373,0.688


*Como ler:* um *odds ratio* acima de 1 aumenta a chance de inadimplência e abaixo de 1 reduz. As variáveis numéricas estão padronizadas, então o efeito é "por 1 desvio-padrão". Deixamos as cidades de fora da tabela: cada uma tem poucos contratos, e os coeficientes delas oscilam muito (são mais ruído do que padrão).

### Extra · Escolha do limiar (custo do erro na prática)
O limiar padrão é 0,5. Como o falso negativo é mais caro, o banco pode **baixar o limiar** para capturar mais inadimplentes, aceitando recusar mais bons pagadores:

In [17]:
proba = modelo_final.predict_proba(X_teste)[:, 1]
linhas = []
for lim in [0.3, 0.4, 0.5, 0.6]:
    p = (proba >= lim).astype(int)
    linhas.append({"limiar": lim, "recall": recall_score(y_teste, p),
                   "precisão": precision_score(y_teste, p, zero_division=0),
                   "% clientes sinalizados": p.mean()})
pd.DataFrame(linhas).round(3)

,limiar,recall,precisão,% clientes sinalizados
0,0.3,0.839,0.195,0.711
1,0.4,0.694,0.225,0.509
2,0.5,0.524,0.267,0.324
3,0.6,0.435,0.370,0.195


No deploy mantemos o limiar **0,5** (o mesmo da avaliação), mas o app mostra a **probabilidade** e uma faixa de risco, para o analista decidir.

## 4 · Deploy do modelo

1. Salvamos o **pipeline completo** (`preparar_entrada` + pré-processamento + regressão logística) com `joblib`, e não apenas o estimador. Assim, o dado novo passa pelas mesmas transformações do treino.
2. Salvamos um `modelo_meta.json` com as métricas, o limiar e as versões.
3. Geramos o `requirements.txt` com as **versões exatas** do ambiente de treino. Um arquivo joblib depende da versão do scikit-learn usada para salvá-lo.
4. Demonstramos o uso: carregamos o arquivo e fazemos uma predição para um cliente novo. A interface completa está no **`app.py` (Streamlit)**.

> ⚠️ **Segurança:** arquivos pickle/joblib podem executar código ao serem carregados. Só carregue modelos de **fonte confiável** (aqui, o arquivo gerado por nós mesmas).

In [18]:
ARQ_MODELO = "modelo_inadimplencia.joblib"
joblib.dump(modelo_final, ARQ_MODELO)

versoes = {"python": platform.python_version(), "scikit-learn": sklearn.__version__,
           "pandas": pd.__version__, "numpy": np.__version__, "joblib": joblib.__version__}
meta = {
    "modelo": melhor,
    "treinado_em": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "limiar": 0.5,
    "random_state": SEMENTE,
    "protocolo": "holdout estratificado 75/25",
    "metricas_teste": {k: round(float(v), 3) for k, v in tabela.loc[melhor, ["ROC-AUC", "PR-AUC", "Recall"]].items()},
    "taxa_inadimplencia_treino": round(float(y_treino.mean()), 3),
    "cidades": sorted(ouro["cidade"].unique().tolist()),
    "versoes": versoes,
}
Path("modelo_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")

Path("requirements.txt").write_text(
    f"scikit-learn=={sklearn.__version__}\n"
    f"pandas=={pd.__version__}\n"
    f"numpy=={np.__version__}\n"
    f"joblib=={joblib.__version__}\n"
    "pyarrow\n"
    "streamlit>=1.50\n", encoding="utf-8")

print(Path("requirements.txt").read_text())
print(f"{ARQ_MODELO}: {Path(ARQ_MODELO).stat().st_size/1024:.0f} KB")

scikit-learn==1.9.1
pandas==3.0.5
numpy==2.5.3
joblib==1.6.0
pyarrow
streamlit>=1.50

modelo_inadimplencia.joblib: 9 KB


In [19]:
# Demonstração: carregar o arquivo salvo e prever para um cliente NOVO
# (como se fosse outro programa: só o arquivo .joblib e o módulo features.py são necessários)
modelo_carregado = joblib.load(ARQ_MODELO)   # fonte confiável: arquivo gerado por nós

novos_clientes = pd.DataFrame([
    {"idade": 42, "sexo": "F", "estado_civil": "Casado", "escolaridade": "Superior",
     "renda_mensal": 6500.0, "empregado": "Sim", "tipo_moradia": "Própria", "mudou_recentemente": "Não",
     "num_veiculos": 1, "dependentes": 1, "cidade": "Blumenau", "valor_emprestimo": 8000.0, "prazo": 36},
    {"idade": 23, "sexo": "M", "estado_civil": "solteiro ", "escolaridade": "Fundamental",   # grafia "suja" de propósito
     "renda_mensal": 1400.0, "empregado": "Não", "tipo_moradia": "Alugada", "mudou_recentemente": "Sim",
     "num_veiculos": 0, "dependentes": 3, "cidade": "Joinville", "valor_emprestimo": 15000.0, "prazo": 1},  # prazo em anos
])

prob = modelo_carregado.predict_proba(novos_clientes)[:, 1]
for i, p in enumerate(prob, 1):
    print(f"Cliente {i}: probabilidade de inadimplência = {p:.1%} → "
          f"{'ALTO RISCO' if p >= meta['limiar'] else 'baixo risco'}")

Cliente 1: probabilidade de inadimplência = 14.9% → baixo risco
Cliente 2: probabilidade de inadimplência = 99.3% → ALTO RISCO


O cliente 2 foi digitado com `estado_civil = "solteiro "` e `prazo = 1` (em anos), e o pipeline corrigiu tudo sozinho, porque as regras de limpeza estão **dentro** dele.

### Interface Streamlit

Com os arquivos `app.py`, `features.py`, `modelo_inadimplencia.joblib`, `modelo_meta.json` e `requirements.txt` na mesma pasta:

```bash
pip install -r requirements.txt
streamlit run app.py
```

O app abre no navegador. O analista preenche os dados do cliente e vê a probabilidade de inadimplência, a faixa de risco e o comprometimento de renda.

## Conclusão

- **Domínio:** previsão de inadimplência em crédito (classificação binária desbalanceada). O falso negativo é o erro mais caro.
- **Pipeline:** Lakehouse bronze → prata → ouro, com a limpeza e os atributos aplicados dentro do pipeline, sem vazamento (sem `qtd_cobrancas`, pré-processamento ajustado só no treino).
- **Modelo escolhido:** Regressão Logística, a melhor nas três métricas, mais estável e interpretável.
- **Deploy:** pipeline completo salvo com joblib e servido numa interface Streamlit.
- **Próximos passos:** calibrar o limiar pelo custo real de cada erro, monitorar *drift* dos dados e testar mais variáveis (histórico de crédito, bureau).